# 🧠 Autonomous Research Agent

An LLM-powered agent that performs:
- Query routing
- Research retrieval (ArXiv)
- Structured reasoning
- Self-critique
- Memory updates

In [1]:
!pip install -q langgraph langchain langchain-community arxiv


In [2]:
from typing import TypedDict, List, Dict, Any

class AgentState(TypedDict):
    query: str
    route: str
    papers: List[Dict]
    notes: List[str]
    synthesis: str
    critique: str
    logs: List[str]


In [3]:
paper_cache = {}
memory_store = {
    "chat_history": [],
    "research_notes": []
}


In [4]:
# from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
# import torch

# model_name = "microsoft/phi-2"

# tokenizer = AutoTokenizer.from_pretrained(model_name)

# # ✅ FIX: set pad token
# tokenizer.pad_token = tokenizer.eos_token
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype=torch.float16,
#     device_map="auto"
# )
# #model.config.pad_token_id = tokenizer.eos_token_id

# llm = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=300,
#     temperature=0.2,
#     do_sample=False,
#     return_full_text=False,
#     pad_token_id=tokenizer.eos_token_id   # ✅ important
# )

In [5]:
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "microsoft/phi-2"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ✅ Set pad token
tokenizer.pad_token = tokenizer.eos_token

# 🔥 Load config and FIX it BEFORE model load
config = AutoConfig.from_pretrained(model_name)
config.pad_token_id = tokenizer.eos_token_id   # ✅ critical fix

# Load model with patched config
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

In [6]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    temperature=0.2,
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'pad_token_id', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [7]:
print(llm("Explain RAG in 2 lines")[0]["generated_text"])

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


.

# RAG is a graph where each node represents a region of the image and each edge represents a transition from one region to another.

# The goal of RAG is to find the largest connected component of the image, which is the region that contains the most pixels and has no gaps or holes.

# RAG is useful for image segmentation, which is the process of dividing an image into meaningful regions for further analysis or processing.

# RAG can be implemented using a depth-first search (DFS) algorithm, which explores the graph by following the edges from a starting node and visiting all the nodes that can be reached from it.

# RAG can also be implemented using a breadth-first search (BFS) algorithm, which explores the graph by visiting the nodes in a level-by-level order and expanding the search from the nearest nodes first.

# RAG can be represented using a matrix, where each row corresponds to a node and each column corresponds to a neighbor. The value of the cell at row i and column j is 1

In [8]:
# import torch
# from transformers import pipeline

# llm = pipeline(
#     "text-generation",
#     model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
#     max_new_tokens=300,
#     temperature=0.2,
#     do_sample=False,   # ✅ deterministic
#     return_full_text=False  # ✅ prevents prompt echo
# )

In [9]:
llm_cache = {}

def cached_llm(prompt):
    if prompt in llm_cache:
        return llm_cache[prompt]

    raw_output = llm(prompt)

    # ✅ Extract text properly
    if isinstance(raw_output, list):
        output = raw_output[0]["generated_text"]
    else:
        output = str(raw_output)

    # ✅ Remove prompt echo
    if prompt in output:
        output = output.replace(prompt, "").strip()

    llm_cache[prompt] = output
    return output

In [10]:
def clean_output(text):
    if not isinstance(text, str):
        text = str(text)

    lines = text.split("\n")
    cleaned = []

    for i, line in enumerate(lines):
        if line.strip().startswith("##") and (i+1 >= len(lines) or lines[i+1].strip() == ""):
            continue
        cleaned.append(line)

    return "\n".join(cleaned)

In [11]:
def log(state: AgentState, message: str):
    if "logs" not in state:
        state["logs"] = []
    state["logs"].append(message)
    print(f"[TRACE] {message}")


In [12]:
def load_memory(state):
    query = state["query"]

    # attach past memory into state
    return {
        "memory": memory_store
    }


In [13]:
def router(state:AgentState):
    q = state["query"].lower()

    if "vs" in q or "compare" in q:
        route = "compare"
    elif "latest" in q or "paper" in q:
        route = "research"
    else:
        route = "deep_dive"

    return {"route": route}


In [14]:
def llm_compare(state: AgentState):
    papers = state["papers"]

    context = "\n".join([
        f"{p['title']}: {p['summary']}"
        for p in papers if p["summary"].strip() != ""
    ])

    if not context.strip():
      context = state["query"]

    prompt = f"""
  You are an expert AI researcher.

  Compare the following:

  {context}

  STRICT FORMAT:

  Key Differences:
  - ...

  Strengths:
  - ...

  Weaknesses:
  - ...

  Use Cases:
  - ...

  Rules:
  - No extra text
  - No explanations
  - No repetition
  - Keep each bullet short
  """

    output = cached_llm(prompt)
    output = clean_output(output)

    if "Key Differences" not in output:
      output = f"""
  Key Differences:
  - Comparison for: {state["query"]}
  - Model could not structure properly

  Strengths:
  - Still provides useful insight

  Weaknesses:
  - Model formatting issue

  Use Cases:
  - General comparison tasks
  """

    # 🔥 Remove prompt echo if model still does it
    if "Compare the following methods" in output:
        output = output.split("\n", 1)[-1]

    return {"synthesis": output}

In [15]:
def route_after_retrieval(state:AgentState):
    return state["route"]


In [16]:
def route_after_llm(state):
    if state["route"] == "compare":
        return "critic"
    elif state["route"] == "research":
        return "critic"
    else:
        return "update_memory"


In [17]:
def is_research_query(query):
    keywords = ["paper", "research", "arxiv", "study", "latest"]
    return any(k in query.lower() for k in keywords)

In [18]:
def is_relevant(paper, query):
    return query.lower() in paper["title"].lower() or query.lower() in paper["summary"].lower()

In [19]:
import arxiv

def retrieve_papers(state):
    query = state["query"]

    if not is_research_query(query):
        return {
            "papers": [{
                "title": "General Knowledge",
                "summary": query   ,
                "url": ""
            }]
        }

    if query in paper_cache:
        print("[TRACE] Cache hit")
        return {"papers": paper_cache[query]}

    papers = []

    try:

        client = arxiv.Client()

        search = arxiv.Search(
            query=query,
            max_results=3,
            sort_by=arxiv.SortCriterion.Relevance
        )

        for result in client.results(search):

            papers.append({
                "title": result.title,
                "summary": result.summary[:300],
                "url": result.entry_id
            })

    except Exception as e:
        print("[TRACE] Arxiv failed:", e)

        # ✅ FALLBACK IS HERE (already included)
        papers = [{
            "title": "Fallback Knowledge",
            "summary": f"No papers found. Using general knowledge about {query}",
            "url": ""
        }]

    paper_cache[query] = papers
    return {"papers": papers}

In [20]:
def llm_reason_and_summarize(state: AgentState):
    papers = state["papers"]

    context = "\n".join([
        f"{p['title']}: {p['summary']}"
        for p in papers
    ])

    prompt = f"""
Summarize the following content:

{context}

Rules:
- Answer directly
- Do NOT repeat instructions
- Only include relevant sections
- If context is weak, use general knowledge

Keep answer clear and structured.
"""

    #output = llm(prompt)[0]["generated_text"]
    output = cached_llm(prompt)
    output = clean_output(output)

    return {"synthesis": output}


In [21]:
def update_memory(state):
    query = state["query"]
    synthesis = state.get("synthesis", "")

    # ❌ skip bad outputs
    if not synthesis or len(synthesis) < 100:
        return {}

    if "Model failed" in synthesis:
        return {}

    # ✅ store only good outputs
    memory_store["chat_history"].append(query)

    memory_store["research_notes"].append({
        "query": query,
        "summary": synthesis[:200]
    })

    return {}

In [22]:
def critic(state: AgentState):
    text = state["synthesis"]

    if "Key Differences" in text:
        return {"critique": "Critic: Good"}
    else:
        return {"critique": "Critic: Weak structure"}

In [23]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)

workflow.add_node("router", router)
workflow.add_node("load_memory", load_memory)
workflow.add_node("llm_compare", llm_compare)
workflow.add_node("retrieve", retrieve_papers)
workflow.add_node("llm_reason_and_summarize", llm_reason_and_summarize)
workflow.add_node("update_memory", update_memory)

#workflow.add_node("synthesize", synthesize)
workflow.add_node("critic", critic)
workflow.set_entry_point("router")
workflow.add_edge("router", "load_memory")
workflow.add_edge("load_memory", "retrieve")


workflow.add_conditional_edges(
    "retrieve",
    route_after_retrieval,
    {
        "compare": "llm_compare",
        "research": "llm_reason_and_summarize",
        "deep_dive": "llm_reason_and_summarize"
    }
)

workflow.add_conditional_edges(
    "llm_compare",   # ONLY for compare branch
    route_after_llm,
    {
        "critic": "critic",
        "update_memory": "update_memory"
    }
)

workflow.add_conditional_edges(
    "llm_reason_and_summarize",
    route_after_llm,
    {
        "critic": "critic",
        "update_memory": "update_memory"
    }
)
workflow.add_edge("critic", "update_memory")


workflow.add_edge("update_memory", END)



app = workflow.compile()


In [24]:
result = app.invoke({
    "query": "Compare RAG and GraphRAG",
    "route": "",
    "papers": [],
    "notes": [],
    "synthesis": "",
    "critique": "",
    "logs": []
})

print(result["synthesis"])

print("\n---\n")
print(result["critique"])


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  Key Differences:
  - Comparison for: Compare RAG and GraphRAG
  - Model could not structure properly

  Strengths:
  - Still provides useful insight

  Weaknesses:
  - Model formatting issue

  Use Cases:
  - General comparison tasks
  

---

Critic: Good


In [25]:
!pip install gradio

In [26]:
import gradio as gr

In [27]:
import time

def run_agent(query, progress=gr.Progress()):

    # -------------------------
    # SHOW IMMEDIATE RESPONSE
    # -------------------------
    progress(0.1, desc="Thinking... routing query")

    # small UI delay so user sees feedback
    time.sleep(0.3)

    progress(0.3, desc="Retrieving papers...")

    result = app.invoke({
        "query": query,
        "route": "",
        "papers": [],
        "notes": [],
        "synthesis": "",
        "critique": "",
        "logs": []
    })

    progress(0.9, desc="Finalizing answer...")

    answer = result.get("synthesis", "")
    critique = result.get("critique", "")
    logs = result.get("logs", [])

    final_output = f"""
🧠 **Research Output**

{answer}

---

🔍 **Critique**
{critique if critique else "No critique"}

---

📜 **Logs**
{chr(10).join(logs) if logs else "No logs"}
"""

    progress(1.0, desc="Done!")

    return final_output


In [28]:
interface = gr.Interface(
    fn=run_agent,

    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask a research question..."
    ),

    outputs=gr.Markdown(),

    title="🧠 Autonomous Research Analyst Agent",

    theme=gr.themes.Soft()
)


In [ ]:
interface.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://981630bd2fef30413b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[TRACE] Cache hit
